# 01 · CSIC 2010 baseline (TF-IDF + Logistic Regression)

Goal: load CSIC 2010, clean it, and train a first model that separates normal requests from attacks.

We try two versions of the text:
- **full**: Method + URL + body (everything CSIC has)
- **log**: Method + URL only, i.e. what a standard Nginx access log records. This shows how much we lose without the body.

In [ ]:
from pathlib import Path
from urllib.parse import unquote_plus
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

# the notebook lives in notebooks/, so the data folder is one level up
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
RAW = ROOT / 'data' / 'raw' / 'csic_database.csv'
OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)
RAW

## 1. Load and clean

- `Unnamed: 0` holds Normal/Anomalous and `classification` holds 0/1. We check they match and use `classification` as the label.
- Strip ` HTTP/1.1` from the end of the URL and `http://localhost:8080` from the start, since an access log only records the path.
- URL-decode the text so the model sees the real characters.

In [ ]:
df = pd.read_csv(RAW)
df = df.rename(columns={'Unnamed: 0': 'label_text', 'classification': 'label', 'lenght': 'length'})
print(df.shape)
print(pd.crosstab(df['label_text'], df['label']))

In [ ]:
def clean_url(u):
    u = str(u)
    if u.endswith(' HTTP/1.1'):
        u = u[:-len(' HTTP/1.1')]
    u = u.replace('http://localhost:8080', '', 1)
    return u

def decode(s):
    return unquote_plus(str(s), encoding='latin-1')

df['path'] = df['URL'].map(clean_url)
df['body'] = df['content'].fillna('')
df['text_log'] = df['Method'] + ' ' + df['path'].map(decode)
df['text_full'] = df['text_log'] + ' ' + df['body'].map(decode)
df[['Method', 'path', 'body', 'label']].head()

## 2. Quick look

In [ ]:
print(df['label'].value_counts(normalize=True).round(3))
print(df.groupby('Method')['label'].agg(['count', 'mean']).round(3))
dups = df.duplicated(subset=['text_full']).sum()
conflict = df.groupby('text_full')['label'].nunique().gt(1).sum()
print(f'duplicate requests: {dups}, same text with both labels: {conflict}')

**Note:** with duplicate requests, the same request can land in both train and test, which inflates the score. So we drop duplicates before splitting, and drop any request that appears with both labels.

In [ ]:
conflict_texts = df.groupby('text_full')['label'].nunique().loc[lambda s: s > 1].index
clean = df[~df['text_full'].isin(conflict_texts)].drop_duplicates(subset=['text_full']).reset_index(drop=True)
print(len(df), '->', len(clean))
print(clean['label'].value_counts())
clean[['Method', 'path', 'body', 'text_log', 'text_full', 'label']].to_csv(OUT / 'csic_clean.csv', index=False)

## 3. Baseline

TF-IDF on characters (n-grams of 2 to 4), because a URL is not normal text and has no spaces. `class_weight='balanced'` because the two classes are not the same size.

In [ ]:
train_df, test_df = train_test_split(clean, test_size=0.2, stratify=clean['label'], random_state=42)

def baseline():
    return make_pipeline(
        TfidfVectorizer(analyzer='char', ngram_range=(2, 4), min_df=2, max_features=100_000, sublinear_tf=True),
        LogisticRegression(max_iter=2000, class_weight='balanced'),
    )

models, results = {}, {}
for col in ['text_full', 'text_log']:
    m = baseline().fit(train_df[col], train_df['label'])
    pred = m.predict(test_df[col])
    models[col] = m
    results[col] = f1_score(test_df['label'], pred)
    print(f'===== {col} =====')
    print(classification_report(test_df['label'], pred, target_names=['Normal', 'Anomalous'], digits=3))

pd.Series(results, name='F1 (Anomalous)').round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, models):
    ConfusionMatrixDisplay.from_estimator(models[col], test_df[col], test_df['label'],
                                          display_labels=['Normal', 'Anomalous'], ax=ax, colorbar=False)
    ax.set_title(col)
plt.tight_layout()

## 4. What does the model look at?

The n-grams that push most towards Anomalous and towards Normal (`text_log` model).

In [ ]:
vec, lr = models['text_log'].named_steps.values()
coef = pd.Series(lr.coef_[0], index=vec.get_feature_names_out())
pd.DataFrame({'anomalous': coef.nlargest(20).index, 'normal': coef.nsmallest(20).index})

## 5. Notes

- What is F1 for full and for log? The gap is what we lose with a standard access log.
- POST requests: the log version has no body, so attacks inside the body are invisible. This goes in the report; our lab logs the body for this reason.
- Next: compare LR, RF and XGBoost with cross-validation (notebook 02).